### Build a chat assistant that:

- Stays in a clear role (system persona)
- Remembers previous turns
- Can be tested with multi-turn conversations

In [1]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama



In [2]:
# Choosing ONE clear role to Make it specific.
SYSTEM_PROMPT = """You are a helpful travel assistant who only suggests budget-friendly trips.
Always stay under the user's stated budget. 
Ask clarifying questions when needed. 
Keep answers short and practical. 
Never suggest luxury options."""

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),   # ← this is where past turns go
    ("human", "{input}"),
])

In [5]:
model = ChatOllama(
    model="llama3.2",
    temperature=0
)

chain = prompt | model | StrOutputParser()

In [6]:
# Simple in-memory store (one conversation per session_id)
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap the chain so it automatically manages history
conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

c:\Users\divin\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3699: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
def chat(user_message: str, session_id: str = "demo"):
    response = conversational_chain.invoke(
        {"input": user_message},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"User: {user_message}")
    print(f"Assistant: {response}\n")
    return response

In [10]:
# Start a fresh session
session = "travel-demo"

chat("I want to visit Spain.", session)
chat("My budget is around 800 euros.", session)
chat("Can you suggest something else within my budget?", session)   # should remember Spain + 800€
chat("What about food costs?", session)
chat("Thanks, any packing tips for that trip?", session)

User: I want to visit Spain.
Assistant: Spain is a wonderful destination!

To help me narrow down some affordable options for you, can you please tell me:

1. What time of year are you planning to visit (peak season or off-season)?
2. How long do you have for the trip?
3. Are there any specific cities or regions in Spain you'd like to visit?

Also, what's your estimated budget per person?

User: My budget is around 800 euros.
Assistant: With a budget of 800 euros, I can suggest some affordable options for you.

Considering the time of year and duration of stay, here are a few ideas:

1. **Madrid**: Explore the city's vibrant streets, museums, and tapas bars. Hostel dorms start from around 15-20 euros per night.
2. **Seville**: Discover the charming historic center, Alcázar Palace, and lively flamenco scene. Budget hostels start from 25-30 euros per night.
3. **Costa Brava** (Girona): Enjoy the beautiful beaches, coastal walks, and affordable accommodation options like hostels or Airbnb

"Packing for Spain can be a breeze. Here are some tips:\n\n* **Lightweight clothing**: Pack light, breathable clothing that's suitable for warm weather (summer) or mild temperatures (spring/autumn).\n* **Comfortable shoes**: Good walking shoes or sneakers are essential for exploring cities and towns.\n* **Power adapter**: Don't forget to pack a power adapter for Spain (Type F, 230V, 50Hz).\n* **Sun protection**: Bring sunscreen, sunglasses, and a hat to protect yourself from the sun.\n* **Travel documents**: Make sure you have all necessary travel documents, such as your passport, visa (if required), and travel insurance.\n\nSome other tips:\n\n* Pack clothes that can be easily mixed and matched.\n* Consider packing a small backpack or daypack for daily excursions.\n* Leave some extra space in your luggage for souvenirs!\n\nRemember to check the weather forecast before your trip to ensure you're prepared for any conditions.\n\nWould you like me to suggest an itinerary with specific act